<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/07-finetune_llm.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

## Utility cells

During the course, we'll leverage some course utilities to streamline our workflow. These utilities are located in the `course` package, which can simply be imported given that we installed the project from git repository above.
You can find the source code [here](https://github.com/PrunaAI/ai-efficiency-courses/tree/main/course).

These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS, MEDIUM_MODEL_IDS, LARGE_MODEL_IDS, ALL_MODEL_IDS

MODEL_IDS = SMALL_MODEL_IDS
# MODEL_IDS = MEDIUM_MODEL_IDS
# MODEL_IDS = LARGE_MODEL_IDS
# MODEL_IDS = ALL_MODEL_IDS

MODEL_IDS

We also recommend to set a custom cache directory for models. Loading models can take significant disk space. To avoid filling up your default disk, we recommend setting a custom cache directory for downloaded models. You can do this by running the following in a terminal or in a notebook cell:

In [ ]:
# Replace <path_to_cache> with your desired cache path
import os

CACHE_PATH = "<path_to_cache>"
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH
os.environ["TRANSFORMERS_CACHE"] = CACHE_PATH

You can also clear the cache by running the following cell:

In [ ]:
from course.models import clear_cache

clear_cache("<path_to_cache>")

# 07: Finetuning to Recover Performance of Quantized LLMs

Welcome to the seventh lecture of the AI Efficiency course! 🚀

In this tutorial, we'll dive into the process of finetuning LLMs after having applied quantization. This can be done with the so-called Parameter-Efficient Retraining after Pruning (PERP).

By the end of this lecture, you will:
- Understand the basics and importance of recovering the performance of quantized LLMs.
- Learn how to prepare data and configure training for PERP.
- Evaluate and analyze the results of your finetuned models.
- Gain hands-on experience with practical tools and efficient workflows for PERP.

Let's get started and see how you can make quantized LLMs work even better for your needs!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `torch`, `transformers` and `pruna` for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using `matplotlib` for basic plotting. Feel free to add any other libraries you might need.

In [ ]:
import copy
import random

from pruna import SmashConfig, smash
from datasets import load_dataset
from transformers import AutoModelForCausalLM
from pruna.data.utils import split_train_into_train_val_test
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.evaluation.metrics import (
    TotalTimeMetric,
    TorchMetricWrapper,
)

Multiple distributions found for package optimum. Picked distribution: optimum


From the `course` package, we will be providing the `evaluate_model` function. This function takes a model id or an optimized `PrunaModel` and evaluates it using the a list of `PrunaMetric`s and a `PrunaDataModule`. If you don't want to provide any metrics of PrunaDataModule, the function will use a set of default metrics and the `WikiText` dataset.

Let's load our first model and evaluate it. For now, we will not provide any metrics or evaluation dataset, but we'll get to that later.

In [ ]:
from course import evaluate_model

### To Complete ###

### 2. Benchmarking LLM Recovery Methods

In this section, you'll learn how to systematically benchmark different PERP finetuning strategies for LLMs after quantization.

As you complete this section, refer to the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/configure.html) for detailed guidance on AI efficiency algorithms and defining smash configurations.

### 2.1 Benchmarking Base Model Quality

In this section, you'll evaluate the base LLM's efficiency and performance using the WikiText dataset.

**Tasks:**
1. Set up an evaluation to measure both perplexity (accuracy) and inference latency (efficiency) for the base model on WikiText.
2. Optionally, repeat the experiment with different LLMs or datasets to compare results.

**Why is this important?**
Benchmarking the unmodified model provides a reference point for later experiments with quantization and finetuning. By measuring both perplexity and latency, you can assess the trade-offs between model quality and speed.

**Instructions:**
- Use the provided `evaluate_model` function from the `course` package.
- Instead of default settings, explicitly define the metrics (perplexity and latency) and dataset (WikiText) to pass as arguments.
- Observe and discuss how the base model's accuracy and efficiency might vary with different models or datasets.

In [ ]:
def smash_evaluate_perplexity_time(model, tokenizer, smash_config=None):
    ### To Complete ###

### 2.2 Finetuning with In-Distribution Data

In this section, you'll explore how finetuning a quantized LLM with in-distribution data (WikiText) can impact both model quality and efficiency.

**Your tasks:**
1. Apply Quanto quantization to the base model using a `SmashConfig` object.
2. Finetune the quantized model in-place.
3. Evaluate its perplexity (accuracy) and inference latency (efficiency).
4. Add the WikiText dataset to the `SmashConfig` object.
5. Compare the results to the unmodified and quantized-only baselines.

**Key questions to consider:**
- Does finetuning improve the model's performance after quantization?
- How does finetuning affect inference latency, and what might explain any observed changes?

As you work through this, observe the trade-offs between model quality and speed, and discuss your findings.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0])
tokenizer = MODEL_IDS[0]

### To Complete ###

### 2.3 Finetuning LLMs with Varying Amounts of In-Distribution Data

In this section, you'll investigate how the amount of in-distribution data used for finetuning a quantized LLM (using Quanto) affects both model quality and efficiency.

**Your task:**
- Finetune a quantized LLM (in-place or by adding parameters) with different sizes of WikiText data, then evaluate its perplexity and inference latency.
- Load the WikiText dataset from `mikasenghaas/wikitext-2` and split it into train, validation, and test sets.
- Select the number of rows from the train set you want to use for training, testing and validation.
- Add the dataset to the `SmashConfig` object.

**Key questions:**
- Does increasing the amount of finetuning data improve model performance?
- How does the amount of data used for finetuning impact inference latency, and why?

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0])
tokenizer = MODEL_IDS[0]

### To Complete ###

### 2.4 Finetuning LLMs with Random Data

Now we've seen how training size influences the performance of a finetuned model, you'll explore how finetuning a quantized LLM with random (out-of-distribution) data affects model quality and inference latency.

**Your task:**
- Finetune a quantized LLM (in-place or by adding parameters) using randomly generated text data, then evaluate its perplexity and inference latency on the WikiText dataset.
- Generate random data.
- Add the dataset to the `SmashConfig` object.
- Evaluate the model's perplexity and inference latency.

**Key questions:**
- Does finetuning with random data lead to any performance improvement?
- How does this affect inference latency, and why?

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0])
tokenizer = MODEL_IDS[0]

### To Complete ###

### 3.4 Finetuning LLMs with Out-of-Distribution Data

In this section, you'll investigate how finetuning a quantized large language model (LLM) with out-of-distribution (OOD) data—such as text from a different domain—impacts both model quality and inference latency.

**Your task:**
- Finetune a quantized LLM (using Quanto, either in-place or by adding parameters) with OOD data (e.g., BookCorpus), and then evaluate its perplexity and inference latency on the WikiText dataset.
- Load a dataset, e.g. the BookCorpus dataset from `SamuelYang/bookcorpus` and split it into train, validation, and test sets.
- Prepare and add the OOD dataset to the `SmashConfig` object.
- Measure and compare the model's performance before and after finetuning.

**Key questions:**
- Does finetuning with out-of-distribution data lead to any performance improvement on WikiText?
- How does this affect inference latency, and what might explain any observed changes?

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0])
tokenizer = MODEL_IDS[0]

### To Complete ###

## Congratulations!

You've reached the end of the course! 🎉 

Now that you've mastered the basics of LLM efficiency, we recommend you to review the course content and try to apply the concepts you've learned to your own projects. We've also included some bonus sections and created several projects in the [course repository](https://github.com/PrunaAI/ai-efficiency-courses) that you can try to complete! In case you are interested in contributing to the course, please reach out to us too.

## ⭐ Bonus Exercise: Analyzing PERP Methods Across Model Architectures

As a bonus, you can try applying the same PERP method across different model architectures. It could be interesting to see how the performance of the different PERP methods varies between models and whether some methods are more robust to the model size or to variations in the recovery dataset.

**Tasks:**
1. For each model architecture in `MODEL_IDS`, repeat the PERP configuration and evaluation process.
2. Compare the results across different model architectures and PERP methods.

**Key questions:**
- Does the performance of the different PERP methods vary between models?
- Are there any PERP methods that are more robust to the model size or recovery dataset?
- Which PERP method is the best overall?